In [1]:
!pip install datasets
from datasets import load_dataset
import pandas as pd, json, ast

ds = load_dataset("ai4privacy/pii-masking-200k", split="train")
raw = ds.to_pandas()

print(raw.shape)
print(raw.columns.tolist())
raw.head(3)

   ---------------------------------------- 0.0/604.4 kB ? eta -:--:--
   ---------------------------------------- 0.0/604.4 kB ? eta -:--:--
   ----------------- ---------------------- 262.1/604.4 kB ? eta -:--:--
   ---------------------------------------- 604.4/604.4 kB 1.6 MB/s  0:00:00
   ---------------------------------------- 0.0/857.0 kB ? eta -:--:--
   ---------------------------------------- 0.0/857.0 kB ? eta -:--:--
   ---------------------------------------- 0.0/857.0 kB ? eta -:--:--
   ------------------------ --------------- 524.3/857.0 kB 2.4 MB/s eta 0:00:01
   ---------------------------------------- 857.0/857.0 kB 1.6 MB/s  0:00:00
   ---------------------------------------- 0.0/3.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/3.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/3.8 MB ? eta -:--:--
   -- ------------------------------------- 0.3/3.8 MB ? eta -:--:--
   ----- ---------------------------------- 0.5/3.8 MB 932.9 k


[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip
c:\Users\Victus\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\Victus\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Victus\.cache\huggingface\hub\datasets--ai4privacy--pii-masking-200k. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/hugg

(209261, 9)
['source_text', 'target_text', 'privacy_mask', 'span_labels', 'mbert_text_tokens', 'mbert_bio_labels', 'id', 'language', 'set']


,source_text,target_text,privacy_mask,span_labels,mbert_text_tokens,mbert_bio_labels,id,language,set
0,A student's assessment was found on device bea...,A student's assessment was found on device bea...,"[{'value': '06-184755-866851-3', 'start': 57, ...","[[0, 57, ""O""], [57, 75, ""PHONEIMEI""], [75, 138...","[A, student, ', s, assessment, was, found, on,...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, B-P...",165761,en,train
1,"Dear Omer, as per our records, your license 78...","Dear [FIRSTNAME], as per our records, your lic...","[{'value': 'Omer', 'start': 5, 'end': 9, 'labe...","[[0, 5, ""O""], [5, 9, ""FIRSTNAME""], [9, 44, ""O""...","[Dear, Omer, ,, as, per, our, records, ,, your...","[O, B-FIRSTNAME, O, O, O, O, O, O, O, O, B-VEH...",165762,en,train
2,Kattie could you please share your recomndatio...,[FIRSTNAME] could you please share your recomn...,"[{'value': 'Kattie', 'start': 0, 'end': 6, 'la...","[[0, 6, ""FIRSTNAME""], [6, 75, ""O""], [75, 77, ""...","[Kat, ##tie, could, you, pl, ##eas, ##e, share...","[B-FIRSTNAME, I-FIRSTNAME, O, O, O, O, O, O, O...",165763,en,train


In [2]:
def parse_mask(m):
    if isinstance(m, str):
        try: return json.loads(m)
        except: return ast.literal_eval(m)
    return m

raw["mask"] = raw["privacy_mask"].apply(parse_mask)
all_labels = set(item["label"] for m in raw["mask"] for item in m)
print(sorted(all_labels))

['ACCOUNTNAME', 'ACCOUNTNUMBER', 'AGE', 'AMOUNT', 'BIC', 'BITCOINADDRESS', 'BUILDINGNUMBER', 'CITY', 'COMPANYNAME', 'COUNTY', 'CREDITCARDCVV', 'CREDITCARDISSUER', 'CREDITCARDNUMBER', 'CURRENCY', 'CURRENCYCODE', 'CURRENCYNAME', 'CURRENCYSYMBOL', 'DATE', 'DOB', 'EMAIL', 'ETHEREUMADDRESS', 'EYECOLOR', 'FIRSTNAME', 'GENDER', 'HEIGHT', 'IBAN', 'IP', 'IPV4', 'IPV6', 'JOBAREA', 'JOBTITLE', 'JOBTYPE', 'LASTNAME', 'LITECOINADDRESS', 'MAC', 'MASKEDNUMBER', 'MIDDLENAME', 'NEARBYGPSCOORDINATE', 'ORDINALDIRECTION', 'PASSWORD', 'PHONEIMEI', 'PHONENUMBER', 'PIN', 'PREFIX', 'SECONDARYADDRESS', 'SEX', 'SSN', 'STATE', 'STREET', 'TIME', 'URL', 'USERAGENT', 'USERNAME', 'VEHICLEVIN', 'VEHICLEVRM', 'ZIPCODE']


In [11]:
if "language" in raw.columns:
    raw = raw[raw["language"].str.lower().isin(["en", "english"])]

def window(text, start, end, pad=40):
    return text[max(0, start - pad): end + pad].strip()

rows = []
for _, r in raw.iterrows():
    text = r["source_text"]
    for item in r["mask"]:
        lab = item["label"].upper()
        if lab == "EMAIL":
            label = "email"
        elif lab == "PHONENUMBER":
            label = "phone"
        elif lab == "PASSWORD":
            label = "credential"
        else:
            continue
        rows.append({
            "text": window(text, item["start"], item["end"]),
            "label": label,
            "source": "ai4privacy"
        })

pub = pd.DataFrame(rows).drop_duplicates(subset="text")
print(pub["label"].value_counts())

label
email         4092
credential    2622
phone         2425
Name: count, dtype: int64


In [12]:
pub = pd.concat(
    [g.sample(min(len(g), 1500), random_state=42) for _, g in pub.groupby("label")]
).reset_index(drop=True)

pub.to_csv("../data/ai4privacy.csv", index=False)
print(pub.shape)
print(pub["label"].value_counts())
pub.groupby("label").sample(4, random_state=1)

(4500, 3)
label
credential    1500
email         1500
phone         1500
Name: count, dtype: int64


,text,label,source
91,ities related to life skills education. vcuFxU...,credential,ai4privacy
75,ease note that the new WIFI password is ynmI8U...,credential,ai4privacy
1264,in with your username and the password: HP85_3...,credential,ai4privacy
330,"ith Elise.Ritchie@hotmail.com, password z_hFZZ...",credential,ai4privacy
2382,tion treatment project. Send resumes to Una_Pa...,email,ai4privacy
1784,Midlands. Please email me the dates at Monserr...,email,ai4privacy
1882,116. Submit any support request through Jonas_...,email,ai4privacy
2680,account with username Frida_Herman and Letha.H...,email,ai4privacy
3042,"er assistance, please dial our helpline +73 73...",phone,ai4privacy
3133,"ions, Marshall, feel free to call us at 078-29...",phone,ai4privacy
